# cNMF Baseline for Topic Modeling Analysis

This notebook provides a baseline implementation using [cNMF (Consensus Non-negative Matrix Factorization)](https://github.com/dylkot/cNMF/) for single-cell topic modeling analysis, extracting interpretable topics from gene expression data for comparison with scE2TM.

## 📋 Prerequisites

Before running this baseline, please ensure you have:

1. **Followed the main installation guide** in the root directory's README to set up the conda environment

2. **Installed additional dependencies** specific to cNMF:

```bash
# Activate the scE2TM environment first
conda activate scE2TM_env

# Install cNMF
pip install cnmf

In [ ]:
import pandas as pd
import numpy as np
import scanpy as sc
from pathlib import Path
from cnmf import cNMF
import shutil
import warnings
warnings.filterwarnings('ignore')

# ========== Path Configuration ==========
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
DATA_DIR = PROJECT_ROOT / 'data'
OUTPUT_DIR = PROJECT_ROOT / 'output' / 'cNMF'

DATASET_NAME = 'Wang'

# Clean previous results
if OUTPUT_DIR.exists():
    print(f"Cleaning previous output directory: {OUTPUT_DIR}")
    shutil.rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("="*60)
print("Baseline: cNMF Analysis")
print(f"Dataset: {DATASET_NAME}")
print("="*60)

# ========== Load data ==========
print("\n1. Loading data...")
data_path = DATA_DIR / f'{DATASET_NAME}_HIGHPRE.csv'
label_path = DATA_DIR / f'{DATASET_NAME}_cell_anno.csv'

data_df = pd.read_csv(data_path, sep=',', index_col=0)
print(f"   Raw data shape (cells x genes): {data_df.shape}")

# Determine orientation: cNMF expects genes as rows, cells as columns
if data_df.shape[1] > data_df.shape[0]:
    # If more columns than rows, assume columns are genes, rows are cells
    adata = sc.AnnData(data_df.T)  # now rows are genes, columns cells
    print("   Assuming columns are genes, rows are cells. Transposed.")
else:
    # If more rows than columns, assume rows are genes, columns cells
    adata = sc.AnnData(data_df)   # already genes rows, cells columns
    print("   Assuming rows are genes, columns are cells. No transpose needed.")

# Load labels if available (labels are per cell, so match adata.obs_names)
if label_path.exists():
    label_df = pd.read_csv(label_path, sep=',', index_col=0)
    common_cells = adata.obs_names.intersection(label_df.index)
    if len(common_cells) > 0:
        adata = adata[common_cells, :]   # subset cells
        adata.obs['cell_type'] = label_df.loc[common_cells].iloc[:, 0]
        print(f"   Labels loaded: {len(adata.obs['cell_type'].unique())} cell types")

# ========== Save as h5ad ==========
print("\n2. Saving data for cNMF...")
h5ad_path = OUTPUT_DIR / f'{DATASET_NAME}_counts.h5ad'
adata.write(h5ad_path)

# ========== Run cNMF ==========
print("\n3. Running cNMF...")
k = 10
n_iter = 100

cnmf_obj = cNMF(
    output_dir=str(OUTPUT_DIR),
    name=f"{DATASET_NAME}_K{k}"
)

print("   Preparing data...")
cnmf_obj.prepare(
    counts_fn=str(h5ad_path),
    components=[k],
    n_iter=n_iter,
    seed=0
)

print("   Factorizing...")
cnmf_obj.factorize(worker_i=0, total_workers=1)

print("   Combining results...")
cnmf_obj.combine()

print("   Computing consensus matrix...")
cnmf_obj.consensus(k=k, density_threshold=0.01)

print("   Loading results...")
usage, spectra_scores, _, _ = cnmf_obj.load_results(
    K=k,
    density_threshold=0.01
)

# According to user: spectra_scores is cells x topics, usage is genes x topics
cell_topic_matrix = spectra_scores   # cells x topics (theta)
topic_gene_matrix = usage            # genes x topics (beta)

# Convert to numpy arrays to avoid DataFrame indexing issues
if hasattr(cell_topic_matrix, 'values'):
    cell_topic_matrix = cell_topic_matrix.values
if hasattr(topic_gene_matrix, 'values'):
    topic_gene_matrix = topic_gene_matrix.values

print(f"\n   cNMF completed!")
print(f"   Cell-topic matrix (theta) shape: {cell_topic_matrix.shape} (cells x topics)")
print(f"   Topic-gene matrix (beta) shape: {topic_gene_matrix.shape} (genes x topics)")

# ========== Extract correct gene names ==========
print("\n4. Extracting gene names...")
# Determine gene names from original data_df
if topic_gene_matrix.shape[0] == data_df.shape[1]:
    gene_names = data_df.columns.tolist()
    print(f"   Using gene names from data_df columns (genes count: {len(gene_names)})")
elif topic_gene_matrix.shape[0] == data_df.shape[0]:
    gene_names = data_df.index.tolist()
    print(f"   Using gene names from data_df index (genes count: {len(gene_names)})")
else:
    print(f"   Warning: Gene count mismatch. Using generic names.")
    gene_names = [f"Gene_{i}" for i in range(topic_gene_matrix.shape[0])]

# Safety check
if len(gene_names) != topic_gene_matrix.shape[0]:
    print(f"   Still mismatch! Using generic names.")
    gene_names = [f"Gene_{i}" for i in range(topic_gene_matrix.shape[0])]

print(f"   First 5 gene names: {gene_names[:5]}")

# ========== Extract top 20 genes per topic ==========
print("\n5. Extracting top 20 genes per topic...")
topic_top_genes = {}
for topic_idx in range(topic_gene_matrix.shape[1]):   # topics are columns
    scores = topic_gene_matrix[:, topic_idx]          # numpy array
    top_indices = np.argsort(scores)[-20:][::-1]
    # Ensure indices within bounds
    top_indices = [i for i in top_indices if i < len(gene_names)]
    topic_top_genes[topic_idx] = [gene_names[i] for i in top_indices]

print("\n6. Top 20 genes per topic (first 10 shown):")
for i in range(k):
    if i in topic_top_genes:
        print(f"\n   Topic {i}:")
        for j, gene in enumerate(topic_top_genes[i][:10]):
            print(f"      {j+1}. {gene}")
    else:
        print(f"\n   Topic {i}: No genes extracted")
